In [33]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2

!pip install -q mpi4py numba numpy

print("Environment installation completed.")

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6

Environment installation completed.


In [34]:
GITHUB_USERNAME = "Luri1337"

assert (
    GITHUB_USERNAME != "YOUR_GITHUB_USERNAME"
    and len(GITHUB_USERNAME.strip()) > 0
), "Set your real GitHub username."

print(f"Configured student identity: {GITHUB_USERNAME}")

Configured student identity: Luri1337


In [35]:
import sys
import numpy as np
import numba
from numba import cuda
from mpi4py import MPI

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Numba:", numba.__version__)
print("MPI:", MPI.Get_library_version())
print("CUDA available:", cuda.is_available())

if cuda.is_available():
    device = cuda.get_current_device()
    print("GPU:", device.name)
    print("Compute capability:", device.compute_capability)
else:
    raise RuntimeError("CUDA GPU is unavailable!")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 2.4.6
Numba: 0.64.0
MPI: Open MPI v4.1.6, package: Debian OpenMPI, ident: 4.1.6, repo rev: v4.1.6, Sep 30, 2023 
CUDA available: True
GPU: Tesla T4
Compute capability: ComputeCapability(major=7, minor=5)


In [36]:

%%writefile distributed_cuda_solver.py

import os
import sys
import json
import math
import hashlib
import subprocess

import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# ============================================================
# CONSTANTS
# ============================================================

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2

GLOBAL_NX = 512
GLOBAL_NY = 512

LOCAL_NX = 256
LOCAL_NY = 256

STEPS = 400
LAMBDA_DAMP = 0.15

DX = 1.0 / 513.0
DY = 1.0 / 513.0

# ============================================================
# GPU UUID AND STUDENT SEED
# ============================================================

def get_gpu_uuid():
    out = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=uuid",
            "--format=csv,noheader"
        ],
        encoding="utf-8"
    )
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()

    return int(digest[:16], 16)


# ============================================================
# TASK 1: DISTRIBUTED CONTEXT
# ============================================================

def init_distributed_context(github_user):

    comm = MPI.COMM_WORLD

    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError(
            f"Expected exactly 4 MPI ranks, got {size}"
        )

    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)

    instance_seed = derive_student_seed(
        github_user,
        gpu_uuid
    )

    cart_comm = comm.Create_cart(
        dims=[2, 2],
        periods=[False, False],
        reorder=False
    )

    coords = cart_comm.Get_coords(rank)

    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "rank": rank,
        "size": size,
        "coords": coords,
        "neighbors": {
            "up": nbr_up,
            "down": nbr_down,
            "left": nbr_left,
            "right": nbr_right
        },
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed
    }


# ============================================================
# TASK 2: SUBDOMAIN INITIALIZATION
# ============================================================

def initialize_subdomains(context):

    rank = context["rank"]
    row_rank, col_rank = context["coords"]

    seed = context["instance_seed"]

    rng = np.random.default_rng(seed + rank)

    # Local coordinates including ghost cells.
    global_rows = (
        row_rank * LOCAL_NY
        + np.arange(258)
    )

    global_cols = (
        col_rank * LOCAL_NX
        + np.arange(258)
    )

    y = global_rows[:, None] * DY
    x = global_cols[None, :] * DX

    # Spatially heterogeneous diffusivity.
    kappa = (
        0.16
        + 0.04 * np.sin(2 * np.pi * x)
        * np.cos(2 * np.pi * y)
        + 0.02 * np.sin(4 * np.pi * x * y)
    )

    kappa = np.clip(kappa, 0.08, 0.24)

    # Seed-dependent Gaussian thermal pulses.
    u = np.zeros((258, 258), dtype=np.float64)

    pulse_count = 4

    for _ in range(pulse_count):

        center_x = rng.uniform(0.15, 0.85)
        center_y = rng.uniform(0.15, 0.85)

        amplitude = rng.uniform(0.5, 1.0)
        sigma = rng.uniform(0.04, 0.09)

        pulse = amplitude * np.exp(
            -(
                (x - center_x) ** 2
                + (y - center_y) ** 2
            ) / (2 * sigma ** 2)
        )

        u += pulse

    # Localized source field.
    source = np.zeros((258, 258), dtype=np.float64)

    source_x = 0.5
    source_y = 0.5
    source_sigma = 0.06

    source = np.exp(
        -(
            (x - source_x) ** 2
            + (y - source_y) ** 2
        ) / (2 * source_sigma ** 2)
    )

    # Physical Dirichlet boundaries.
    neighbors = context["neighbors"]

    if neighbors["up"] == MPI.PROC_NULL:
        u[0, :] = 0.0

    if neighbors["down"] == MPI.PROC_NULL:
        u[-1, :] = 0.0

    if neighbors["left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0

    if neighbors["right"] == MPI.PROC_NULL:
        u[:, -1] = 0.0

    return {
        "u": u,
        "kappa": kappa,
        "source": source,
        "x": x,
        "y": y
    }


# ============================================================
# INITIALIZATION TEST
# ============================================================

def initialization_test(github_user):

    context = init_distributed_context(github_user)

    fields = initialize_subdomains(context)

    comm = context["cart_comm"]

    print(
        f"Rank {context['rank']} | "
        f"Coordinates: {context['coords']} | "
        f"Neighbors: {context['neighbors']} | "
        f"GPU: {context['gpu_uuid']}"
    )

    print(
        f"Rank {context['rank']} | "
        f"u shape: {fields['u'].shape} | "
        f"kappa range: "
        f"[{fields['kappa'].min():.6f}, "
        f"{fields['kappa'].max():.6f}] | "
        f"Seed: {context['instance_seed']}"
    )

    comm.Barrier()

    if context["rank"] == 0:
        print("\nTask 1 and Task 2 initialization completed.")


if __name__ == "__main__":

    github_user = sys.argv[1]

    initialization_test(github_user)


Overwriting distributed_cuda_solver.py


In [37]:
import os
import subprocess

env = os.environ.copy()

for key in list(env.keys()):
    if (
        key.startswith("OMPI_")
        or key.startswith("PMIX_")
        or key.startswith("PMI_")
        or key.startswith("PRTE_")
    ):
        env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3", "-u",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
    timeout=180
)

print("===== STDOUT =====")
print(proc.stdout)

print("===== STDERR =====")
print(proc.stderr)

print("===== EXIT CODE =====")
print(proc.returncode)

if proc.returncode != 0:
    raise RuntimeError("Solver initialization failed")

===== STDOUT =====
Rank 0 | Coordinates: [0, 0] | Neighbors: {'up': -2, 'down': 2, 'left': -2, 'right': 1} | GPU: GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670
Rank 0 | u shape: (258, 258) | kappa range: [0.140001, 0.201283] | Seed: 3811124356498761665
Rank 2 | Coordinates: [1, 0] | Neighbors: {'up': 0, 'down': -2, 'left': -2, 'right': 3} | GPU: GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670
Rank 2 | u shape: (258, 258) | kappa range: [0.140001, 0.212120] | Seed: 3811124356498761665
Rank 1 | Coordinates: [0, 1] | Neighbors: {'up': -2, 'down': 3, 'left': 0, 'right': -2} | GPU: GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670
Rank 1 | u shape: (258, 258) | kappa range: [0.120000, 0.184639] | Seed: 3811124356498761665Rank 3 | Coordinates: [1, 1] | Neighbors: {'up': 1, 'down': -2, 'left': 2, 'right': -2} | GPU: GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670
Rank 3 | u shape: (258, 258) | kappa range: [0.108040, 0.191833] | Seed: 3811124356498761665


Task 1 and Task 2 initialization completed.

===== STDERR =====


In [38]:

%%writefile -a distributed_cuda_solver.py

# ============================================================
# TASK 3: CUDA HALO PACK / UNPACK
# ============================================================

@cuda.jit
def pack_halos_kernel(u, buffer, n, direction):
    idx = cuda.grid(1)

    if idx < n:
        if direction == 0:       # Top row
            buffer[idx] = u[1, idx + 1]

        elif direction == 1:     # Bottom row
            buffer[idx] = u[n, idx + 1]

        elif direction == 2:     # Left column
            buffer[idx] = u[idx + 1, 1]

        elif direction == 3:     # Right column
            buffer[idx] = u[idx + 1, n]


@cuda.jit
def unpack_halos_kernel(u, buffer, n, direction):
    idx = cuda.grid(1)

    if idx < n:
        if direction == 0:       # Top ghost row
            u[0, idx + 1] = buffer[idx]

        elif direction == 1:     # Bottom ghost row
            u[n + 1, idx + 1] = buffer[idx]

        elif direction == 2:     # Left ghost column
            u[idx + 1, 0] = buffer[idx]

        elif direction == 3:     # Right ghost column
            u[idx + 1, n + 1] = buffer[idx]


# ============================================================
# PINNED HOST BUFFERS AND DEVICE BUFFERS
# ============================================================

def create_halo_buffers(n=256):

    send_host = [
        cuda.pinned_array(n, dtype=np.float64)
        for _ in range(4)
    ]

    recv_host = [
        cuda.pinned_array(n, dtype=np.float64)
        for _ in range(4)
    ]

    send_device = [
        cuda.device_array(n, dtype=np.float64)
        for _ in range(4)
    ]

    recv_device = [
        cuda.device_array(n, dtype=np.float64)
        for _ in range(4)
    ]

    return {
        "send_host": send_host,
        "recv_host": recv_host,
        "send_device": send_device,
        "recv_device": recv_device
    }


# ============================================================
# ASYNCHRONOUS HALO EXCHANGE
# ============================================================

def exchange_halos_async(
    u,
    context,
    buffers,
    stream_boundary
):

    n = LOCAL_NX

    cart_comm = context["cart_comm"]
    neighbors = context["neighbors"]

    send_host = buffers["send_host"]
    recv_host = buffers["recv_host"]

    send_device = buffers["send_device"]
    recv_device = buffers["recv_device"]

    # Pack four owned borders on GPU.
    threads = 256
    blocks = (n + threads - 1) // threads

    for direction in range(4):

        pack_halos_kernel[blocks, threads, stream_boundary](
            u,
            send_device[direction],
            n,
            direction
        )

        send_device[direction].copy_to_host(
            send_host[direction],
            stream=stream_boundary
        )

    # Ensure D2H transfers are complete before MPI reads buffers.
    stream_boundary.synchronize()

    # Directions:
    # 0 = up
    # 1 = down
    # 2 = left
    # 3 = right

    neighbor_list = [
        neighbors["up"],
        neighbors["down"],
        neighbors["left"],
        neighbors["right"]
    ]

    # Opposite directions for incoming halos.
    opposite = [1, 0, 3, 2]

    requests = []

    # Post all receives first.
    for direction in range(4):

        neighbor = neighbor_list[direction]

        if neighbor != MPI.PROC_NULL:

            request = cart_comm.Irecv(
                recv_host[direction],
                source=neighbor,
                tag=opposite[direction]
            )

            requests.append(request)

    # Post all sends.
    for direction in range(4):

        neighbor = neighbor_list[direction]

        if neighbor != MPI.PROC_NULL:

            request = cart_comm.Isend(
                send_host[direction],
                dest=neighbor,
                tag=direction
            )

            requests.append(request)

    # Wait for all communication.
    MPI.Request.Waitall(requests)

    # Transfer received halos to GPU.
    for direction in range(4):

        if neighbor_list[direction] != MPI.PROC_NULL:

            recv_device[direction].copy_to_device(
                recv_host[direction],
                stream=stream_boundary
            )

            unpack_halos_kernel[blocks, threads, stream_boundary](
                u,
                recv_device[direction],
                n,
                direction
            )

    stream_boundary.synchronize()

    return u


# ============================================================
# TASK 3: HALO EXCHANGE TEST
# ============================================================

def halo_exchange_test(github_user):

    context = init_distributed_context(github_user)

    fields = initialize_subdomains(context)

    # Upload local field to GPU.
    u_device = cuda.to_device(fields["u"])

    stream_boundary = cuda.stream()

    buffers = create_halo_buffers(LOCAL_NX)

    # Perform one complete halo exchange.
    exchange_halos_async(
        u_device,
        context,
        buffers,
        stream_boundary
    )

    result = u_device.copy_to_host()

    cart_comm = context["cart_comm"]
    cart_comm.Barrier()

    rank = context["rank"]

    print(
        f"Rank {rank}: "
        f"Halo exchange completed | "
        f"Shape: {result.shape} | "
        f"Top ghost mean: {result[0, 1:-1].mean():.8f} | "
        f"Bottom ghost mean: {result[-1, 1:-1].mean():.8f}",
        flush=True
    )

    cart_comm.Barrier()

    if rank == 0:
        print("Task 3 halo exchange test completed.", flush=True)


Appending to distributed_cuda_solver.py


In [39]:

import os
import subprocess

env = os.environ.copy()

for key in list(env):
    if (
        key.startswith("OMPI_")
        or key.startswith("PMIX_")
        or key.startswith("PMI_")
        or key.startswith("PRTE_")
    ):
        env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "-u",
    "-c",
    (
        "import sys; "
        "import distributed_cuda_solver as s; "
        "s.halo_exchange_test(sys.argv[1])"
    ),
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
    timeout=180
)

print("STDOUT:\n", proc.stdout)
print("STDERR:\n", proc.stderr)
print("EXIT CODE:", proc.returncode)

if proc.returncode != 0:
    raise RuntimeError("Task 3 failed")


STDOUT:
 Rank 0: Halo exchange completed | Shape: (258, 258) | Top ghost mean: 0.00000000 | Bottom ghost mean: 0.00591028Rank 2: Halo exchange completed | Shape: (258, 258) | Top ghost mean: 0.00013861 | Bottom ghost mean: 0.00000000
Rank 3: Halo exchange completed | Shape: (258, 258) | Top ghost mean: 0.18649114 | Bottom ghost mean: 0.00000000
Rank 1: Halo exchange completed | Shape: (258, 258) | Top ghost mean: 0.00000000 | Bottom ghost mean: 0.00012337

Task 3 halo exchange test completed.

STDERR:
 /usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packa

In [40]:

%%writefile -a distributed_cuda_solver.py

# ============================================================
# TASK 4: SHARED MEMORY TILED STENCIL
# ============================================================

@cuda.jit
def tiled_stencil_kernel(
    u,
    kappa,
    source,
    u_next,
    dt,
    dx,
    dy,
    r_start,
    r_end,
    c_start,
    c_end
):

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = r_start + cuda.blockIdx.y * 16 + ty
    col = c_start + cuda.blockIdx.x * 16 + tx

    # Shared memory tile: 16x16 + 1-cell halo apron.
    smem = cuda.shared.array(
        shape=(18, 18),
        dtype=float64
    )

    # Load center cells.
    if row <= 257 and col <= 257:
        smem[ty + 1, tx + 1] = u[row, col]
    else:
        smem[ty + 1, tx + 1] = 0.0

    # Load halo apron.
    if ty == 0:
        if row - 1 >= 0 and col <= 257:
            smem[0, tx + 1] = u[row - 1, col]
        else:
            smem[0, tx + 1] = 0.0

    if ty == 15:
        if row + 1 <= 257 and col <= 257:
            smem[17, tx + 1] = u[row + 1, col]
        else:
            smem[17, tx + 1] = 0.0

    if tx == 0:
        if col - 1 >= 0 and row <= 257:
            smem[ty + 1, 0] = u[row, col - 1]
        else:
            smem[ty + 1, 0] = 0.0

    if tx == 15:
        if col + 1 <= 257 and row <= 257:
            smem[ty + 1, 17] = u[row, col + 1]
        else:
            smem[ty + 1, 17] = 0.0

    # Synchronize all 256 threads.
    cuda.syncthreads()

    # Update only cells inside the requested bounding box.
    if (
        row >= r_start
        and row < r_end
        and col >= c_start
        and col < c_end
        and row >= 1
        and row <= 256
        and col >= 1
        and col <= 256
    ):

        center = smem[ty + 1, tx + 1]

        up = smem[ty, tx + 1]
        down = smem[ty + 2, tx + 1]

        left = smem[ty + 1, tx]
        right = smem[ty + 1, tx + 2]

        k_center = kappa[row, col]

        k_up = 0.5 * (k_center + kappa[row - 1, col])
        k_down = 0.5 * (k_center + kappa[row + 1, col])

        k_left = 0.5 * (k_center + kappa[row, col - 1])
        k_right = 0.5 * (k_center + kappa[row, col + 1])

        diffusion_x = (
            k_down * (down - center)
            - k_up * (center - up)
        ) / (dx * dx)

        diffusion_y = (
            k_right * (right - center)
            - k_left * (center - left)
        ) / (dy * dy)

        damping = 0.15 * center * center * center

        value = center + dt * (
            diffusion_x
            + diffusion_y
            - damping
            + source[row, col]
        )

        u_next[row, col] = value


# ============================================================
# STENCIL LAUNCH HELPER
# ============================================================

def launch_stencil(
    u,
    kappa,
    source,
    u_next,
    dt,
    stream,
    r_start,
    r_end,
    c_start,
    c_end
):

    rows = r_end - r_start
    cols = c_end - c_start

    blocks = (
        math.ceil(cols / 16),
        math.ceil(rows / 16)
    )

    threads = (16, 16)

    tiled_stencil_kernel[blocks, threads, stream](
        u,
        kappa,
        source,
        u_next,
        dt,
        DX,
        DY,
        r_start,
        r_end,
        c_start,
        c_end
    )


Appending to distributed_cuda_solver.py


In [41]:

import numpy as np
from numba import cuda
import importlib
import distributed_cuda_solver as solver

# Reload module to include the latest appended kernels.
importlib.reload(solver)

test_context = {
    "rank": 0,
    "coords": [0, 0],
    "instance_seed": 12345,
    "neighbors": {
        "up": solver.MPI.PROC_NULL,
        "down": 1,
        "left": solver.MPI.PROC_NULL,
        "right": 1
    }
}

fields = solver.initialize_subdomains(test_context)

u_host = fields["u"]
kappa_host = fields["kappa"]
source_host = fields["source"]

u_gpu = cuda.to_device(u_host)
kappa_gpu = cuda.to_device(kappa_host)
source_gpu = cuda.to_device(source_host)

u_next_gpu = cuda.device_array(
    (258, 258),
    dtype=np.float64
)

u_next_gpu.copy_to_device(u_gpu)

stream = cuda.stream()
dt = 1e-7

solver.launch_stencil(
    u_gpu,
    kappa_gpu,
    source_gpu,
    u_next_gpu,
    dt,
    stream,
    2, 256,
    2, 256
)

stream.synchronize()

result = u_next_gpu.copy_to_host()

print("Stencil kernel executed.")
print("Output shape:", result.shape)
print("Finite values:", np.all(np.isfinite(result)))
print("Maximum value:", result.max())
print("Minimum value:", result.min())

print(
    "Interior changed:",
    not np.array_equal(
        result[2:256, 2:256],
        u_host[2:256, 2:256]
    )
)


Stencil kernel executed.
Output shape: (258, 258)
Finite values: True
Maximum value: 1.1328868259427722
Minimum value: 0.0
Interior changed: True


In [42]:
# CPU reference implementation for the same stencil update.

u_cpu = u_host.copy()
reference = u_host.copy()

for i in range(2, 256):
    for j in range(2, 256):

        center = u_cpu[i, j]

        k0 = kappa_host[i, j]

        kup = 0.5 * (k0 + kappa_host[i-1, j])
        kdown = 0.5 * (k0 + kappa_host[i+1, j])
        kleft = 0.5 * (k0 + kappa_host[i, j-1])
        kright = 0.5 * (k0 + kappa_host[i, j+1])

        diffusion_x = (
            kdown * (u_cpu[i+1, j] - center)
            - kup * (center - u_cpu[i-1, j])
        ) / (solver.DX ** 2)

        diffusion_y = (
            kright * (u_cpu[i, j+1] - center)
            - kleft * (center - u_cpu[i, j-1])
        ) / (solver.DY ** 2)

        reference[i, j] = center + dt * (
            diffusion_x
            + diffusion_y
            - 0.15 * center**3
            + source_host[i, j]
        )

difference = np.max(
    np.abs(result[2:256, 2:256] - reference[2:256, 2:256])
)

print("Maximum absolute difference:", difference)
print("Numerical agreement:", difference < 1e-12)

Maximum absolute difference: 5.421010862427522e-20
Numerical agreement: True


In [43]:

%%writefile -a distributed_cuda_solver.py

# ============================================================
# TASK 4: DUAL-STREAM COMPUTE / COMMUNICATION OVERLAP
# ============================================================

def begin_halo_exchange(
    u,
    context,
    buffers,
    stream_boundary
):

    n = LOCAL_NX
    cart_comm = context["cart_comm"]
    neighbors = context["neighbors"]

    send_host = buffers["send_host"]
    recv_host = buffers["recv_host"]

    send_device = buffers["send_device"]

    threads = 256
    blocks = (n + threads - 1) // threads

    # Stage A: pack and asynchronously transfer borders D2H.
    for direction in range(4):

        pack_halos_kernel[blocks, threads, stream_boundary](
            u,
            send_device[direction],
            n,
            direction
        )

        send_device[direction].copy_to_host(
            send_host[direction],
            stream=stream_boundary
        )

    stream_boundary.synchronize()

    neighbor_list = [
        neighbors["up"],
        neighbors["down"],
        neighbors["left"],
        neighbors["right"]
    ]

    opposite = [1, 0, 3, 2]

    requests = []

    # Stage B: post receives.
    for direction in range(4):

        neighbor = neighbor_list[direction]

        if neighbor != MPI.PROC_NULL:

            requests.append(
                cart_comm.Irecv(
                    recv_host[direction],
                    source=neighbor,
                    tag=opposite[direction]
                )
            )

    # Stage B: post sends.
    for direction in range(4):

        neighbor = neighbor_list[direction]

        if neighbor != MPI.PROC_NULL:

            requests.append(
                cart_comm.Isend(
                    send_host[direction],
                    dest=neighbor,
                    tag=direction
                )
            )

    return requests


def finish_halo_exchange(
    u,
    context,
    buffers,
    requests,
    stream_boundary
):

    neighbors = context["neighbors"]

    recv_host = buffers["recv_host"]
    recv_device = buffers["recv_device"]

    neighbor_list = [
        neighbors["up"],
        neighbors["down"],
        neighbors["left"],
        neighbors["right"]
    ]

    # Stage C: wait for MPI transfers.
    MPI.Request.Waitall(requests)

    threads = 256
    blocks = (LOCAL_NX + threads - 1) // threads

    # Transfer received halos H2D and unpack.
    for direction in range(4):

        if neighbor_list[direction] != MPI.PROC_NULL:

            recv_device[direction].copy_to_device(
                recv_host[direction],
                stream=stream_boundary
            )

            unpack_halos_kernel[blocks, threads, stream_boundary](
                u,
                recv_device[direction],
                LOCAL_NX,
                direction
            )


def step_overlapped(
    u,
    u_next,
    kappa,
    source,
    context,
    buffers,
    dt,
    stream_interior,
    stream_boundary
):

    # Start asynchronous halo communication.
    requests = begin_halo_exchange(
        u,
        context,
        buffers,
        stream_boundary
    )

    # Concurrently compute deep interior.
    launch_stencil(
        u,
        kappa,
        source,
        u_next,
        dt,
        stream_interior,
        2, 256,
        2, 256
    )

    # Wait for halo communication and unpack.
    finish_halo_exchange(
        u,
        context,
        buffers,
        requests,
        stream_boundary
    )

    # Wait for unpack operations before boundary computation.
    stream_boundary.synchronize()

    # Compute top perimeter.
    launch_stencil(
        u, kappa, source, u_next,
        dt, stream_boundary,
        1, 2, 1, 257
    )

    # Compute bottom perimeter.
    launch_stencil(
        u, kappa, source, u_next,
        dt, stream_boundary,
        256, 257, 1, 257
    )

    # Compute left perimeter.
    launch_stencil(
        u, kappa, source, u_next,
        dt, stream_boundary,
        2, 256, 1, 2
    )

    # Compute right perimeter.
    launch_stencil(
        u, kappa, source, u_next,
        dt, stream_boundary,
        2, 256, 256, 257
    )

    # Both streams must finish before swapping buffers.
    stream_interior.synchronize()
    stream_boundary.synchronize()

    return u_next


def step_serialized(
    u,
    u_next,
    kappa,
    source,
    context,
    buffers,
    dt,
    stream_boundary
):

    requests = begin_halo_exchange(
        u,
        context,
        buffers,
        stream_boundary
    )

    finish_halo_exchange(
        u,
        context,
        buffers,
        requests,
        stream_boundary
    )

    stream_boundary.synchronize()

    # Update the complete owned interior.
    launch_stencil(
        u,
        kappa,
        source,
        u_next,
        dt,
        stream_boundary,
        1, 257,
        1, 257
    )

    stream_boundary.synchronize()

    return u_next


Appending to distributed_cuda_solver.py


In [44]:

%%writefile -a distributed_cuda_solver.py

def distributed_step_test(github_user):

    context = init_distributed_context(github_user)
    fields = initialize_subdomains(context)

    u = cuda.to_device(fields["u"])
    kappa = cuda.to_device(fields["kappa"])
    source = cuda.to_device(fields["source"])

    u_next = cuda.device_array(
        (258, 258),
        dtype=np.float64
    )

    buffers = create_halo_buffers(LOCAL_NX)

    stream_interior = cuda.stream()
    stream_boundary = cuda.stream()

    dt = 1e-7

    result = step_overlapped(
        u,
        u_next,
        kappa,
        source,
        context,
        buffers,
        dt,
        stream_interior,
        stream_boundary
    )

    host_result = result.copy_to_host()

    context["cart_comm"].Barrier()

    print(
        f"Rank {context['rank']} | "
        f"Step completed | "
        f"Finite: {np.all(np.isfinite(host_result))} | "
        f"Max: {host_result.max():.8f}",
        flush=True
    )


if __name__ == "__main__":

    if len(sys.argv) > 2 and sys.argv[2] == "step_test":
        distributed_step_test(sys.argv[1])
    else:
        initialization_test(sys.argv[1])


Appending to distributed_cuda_solver.py


In [45]:

import os
import subprocess

env = os.environ.copy()

for key in list(env):
    if (
        key.startswith("OMPI_")
        or key.startswith("PMIX_")
        or key.startswith("PMI_")
        or key.startswith("PRTE_")
    ):
        env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "-u",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
    "step_test"
]

proc = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
    timeout=180
)

print("STDOUT:\n", proc.stdout)
print("STDERR:\n", proc.stderr)
print("EXIT CODE:", proc.returncode)

if proc.returncode != 0:
    raise RuntimeError("Distributed step test failed")


STDOUT:
 Rank 3 | Coordinates: [1, 1] | Neighbors: {'up': 1, 'down': -2, 'left': 2, 'right': -2} | GPU: GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670
Rank 3 | u shape: (258, 258) | kappa range: [0.108040, 0.191833] | Seed: 3811124356498761665
Rank 0 | Coordinates: [0, 0] | Neighbors: {'up': -2, 'down': 2, 'left': -2, 'right': 1} | GPU: GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670Rank 2 | Coordinates: [1, 0] | Neighbors: {'up': 0, 'down': -2, 'left': -2, 'right': 3} | GPU: GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670
Rank 0 | u shape: (258, 258) | kappa range: [0.140001, 0.201283] | Seed: 3811124356498761665

Rank 2 | u shape: (258, 258) | kappa range: [0.140001, 0.212120] | Seed: 3811124356498761665
Rank 1 | Coordinates: [0, 1] | Neighbors: {'up': -2, 'down': 3, 'left': 0, 'right': -2} | GPU: GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670
Rank 1 | u shape: (258, 258) | kappa range: [0.120000, 0.184639] | Seed: 3811124356498761665

Task 1 and Task 2 initialization completed.
Rank 1 | Step completed | Fini

In [46]:

%%writefile -a distributed_cuda_solver.py

def parity_test(github_user):

    context = init_distributed_context(github_user)
    fields = initialize_subdomains(context)

    # Identical initial states.
    u_serial = cuda.to_device(fields["u"])
    u_overlap = cuda.to_device(fields["u"])

    kappa = cuda.to_device(fields["kappa"])
    source = cuda.to_device(fields["source"])

    next_serial = cuda.device_array(
        (258, 258), dtype=np.float64
    )

    next_overlap = cuda.device_array(
        (258, 258), dtype=np.float64
    )

    next_serial.copy_to_device(u_serial)
    next_overlap.copy_to_device(u_overlap)

    buffers_serial = create_halo_buffers()
    buffers_overlap = create_halo_buffers()

    stream_serial = cuda.stream()
    stream_interior = cuda.stream()
    stream_boundary = cuda.stream()

    dt = 1e-7

    # Serialized execution.
    step_serialized(
        u_serial,
        next_serial,
        kappa,
        source,
        context,
        buffers_serial,
        dt,
        stream_serial
    )

    # Overlapped execution.
    step_overlapped(
        u_overlap,
        next_overlap,
        kappa,
        source,
        context,
        buffers_overlap,
        dt,
        stream_interior,
        stream_boundary
    )

    serial_result = next_serial.copy_to_host()
    overlap_result = next_overlap.copy_to_host()

    local_diff = np.max(
        np.abs(serial_result - overlap_result)
    )

    global_diff = context["cart_comm"].allreduce(
        local_diff,
        op=MPI.MAX
    )

    print(
        f"Rank {context['rank']} | "
        f"Local difference: {local_diff:.15e}",
        flush=True
    )

    context["cart_comm"].Barrier()

    if context["rank"] == 0:
        print(
            f"\nGlobal maximum difference: {global_diff:.15e}",
            flush=True
        )
        print(
            "Parity passed:",
            global_diff < 1e-12,
            flush=True
        )


Appending to distributed_cuda_solver.py


In [47]:

import os
import subprocess

env = os.environ.copy()

for key in list(env):
    if (
        key.startswith("OMPI_")
        or key.startswith("PMIX_")
        or key.startswith("PMI_")
        or key.startswith("PRTE_")
    ):
        env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "-u",
    "-c",
    (
        "import sys; "
        "import distributed_cuda_solver as s; "
        "s.parity_test(sys.argv[1])"
    ),
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
    timeout=180
)

print("STDOUT:\n", proc.stdout)
print("STDERR:\n", proc.stderr)
print("EXIT CODE:", proc.returncode)

if proc.returncode != 0:
    raise RuntimeError("Parity test failed")


STDOUT:
 Rank 0 | Local difference: 0.000000000000000e+00
Rank 1 | Local difference: 0.000000000000000e+00
Rank 2 | Local difference: 0.000000000000000e+00
Rank 3 | Local difference: 0.000000000000000e+00

Global maximum difference: 0.000000000000000e+00
Parity passed: True

STDERR:
 /usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-

In [48]:

%%writefile -a distributed_cuda_solver.py

# ============================================================
# TASK 5: DISTRIBUTED SOLVER
# ============================================================

def calculate_global_metrics(
    u,
    previous_state,
    cart_comm
):
    host = u.copy_to_host()

    local_energy = (
        np.sum(host[1:257, 1:257])
        * DX * DY
    )

    if previous_state is None:
        local_residual = 0.0
    else:
        delta = (
            host[1:257, 1:257]
            - previous_state
        )

        local_residual = np.sum(delta * delta) * DX * DY

    global_energy = cart_comm.allreduce(
        local_energy,
        op=MPI.SUM
    )

    global_residual = cart_comm.allreduce(
        local_residual,
        op=MPI.SUM
    )

    return (
        float(global_energy),
        float(math.sqrt(global_residual))
    )


def run_mode(
    initial_fields,
    context,
    mode,
    dt=1e-7,
    steps=400
):

    cart_comm = context["cart_comm"]

    u = cuda.to_device(initial_fields["u"])

    u_next = cuda.to_device(initial_fields["u"])

    kappa = cuda.to_device(initial_fields["kappa"])
    source = cuda.to_device(initial_fields["source"])

    buffers = create_halo_buffers(LOCAL_NX)

    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()

    # CUDA event timing.
    start_boundary = cuda.event(timing=True)
    stop_boundary = cuda.event(timing=True)

    start_interior = cuda.event(timing=True)
    stop_interior = cuda.event(timing=True)

    cart_comm.Barrier()

    wall_start = MPI.Wtime()

    start_boundary.record(stream_boundary)
    start_interior.record(stream_interior)

    energy_trace = []
    residual_trace = []

    # Step 0 diagnostics.
    energy, residual = calculate_global_metrics(
        u,
        None,
        cart_comm
    )

    energy_trace.append((0, energy))
    residual_trace.append((0, residual))

    if context["rank"] == 0:
        print(
            f"{mode} | Step 0 | "
            f"Energy={energy:.12e} | "
            f"L2={residual:.12e}",
            flush=True
        )

    for step in range(1, steps + 1):

        # Save the previous timestep only when the next
        # iteration is a checkpoint.
        previous_state = None

        if step % 100 == 0:
            previous_state = u.copy_to_host()[1:257, 1:257].copy()

        if mode == "serialized":

            step_serialized(
                u,
                u_next,
                kappa,
                source,
                context,
                buffers,
                dt,
                stream_boundary
            )

        elif mode == "overlapped":

            step_overlapped(
                u,
                u_next,
                kappa,
                source,
                context,
                buffers,
                dt,
                stream_interior,
                stream_boundary
            )

        else:
            raise ValueError(f"Unknown mode: {mode}")

        # Swap current and next fields.
        u, u_next = u_next, u

        if step % 100 == 0:

            energy, residual = calculate_global_metrics(
                u,
                previous_state,
                cart_comm
            )

            energy_trace.append((step, energy))
            residual_trace.append((step, residual))

            if context["rank"] == 0:
                print(
                    f"{mode} | Step {step} | "
                    f"Energy={energy:.12e} | "
                    f"L2={residual:.12e}",
                    flush=True
                )

    stream_boundary.synchronize()
    stream_interior.synchronize()

    stop_boundary.record(stream_boundary)
    stop_interior.record(stream_interior)

    stop_boundary.synchronize()
    stop_interior.synchronize()

    local_wall_time = MPI.Wtime() - wall_start

    global_wall_time = cart_comm.allreduce(
        local_wall_time,
        op=MPI.MAX
    )

    boundary_ms = start_boundary.elapsed_time(stop_boundary)
    interior_ms = start_interior.elapsed_time(stop_interior)

    local_cuda_ms = max(boundary_ms, interior_ms)

    global_cuda_ms = cart_comm.allreduce(
        local_cuda_ms,
        op=MPI.MAX
    )

    final_state = u.copy_to_host()

    return {
        "final_state": final_state,
        "energy_trace": energy_trace,
        "residual_trace": residual_trace,
        "wall_time_ms": global_wall_time * 1000.0,
        "cuda_event_ms": global_cuda_ms
    }


# ============================================================
# PTX ASSEMBLY AUDIT
# ============================================================

def audit_stencil_ptx():

    dummy = np.zeros((258, 258), dtype=np.float64)

    d_u = cuda.to_device(dummy)
    d_k = cuda.to_device(dummy)
    d_s = cuda.to_device(dummy)
    d_out = cuda.to_device(dummy)

    stream = cuda.stream()

    launch_stencil(
        d_u, d_k, d_s, d_out,
        1e-7, stream,
        2, 256, 2, 256
    )

    stream.synchronize()

    ptx_result = tiled_stencil_kernel.inspect_asm()

    # Support both string and dictionary results.
    if isinstance(ptx_result, dict):
        ptx = "\n".join(
            value for value in ptx_result.values()
            if isinstance(value, str)
        )
    else:
        ptx = ptx_result

    import re

    counts = {
        "st.shared.f64": len(re.findall(r"\bst\.shared\.f64\b", ptx)),
        "ld.shared.f64": len(re.findall(r"\bld\.shared\.f64\b", ptx)),
        "bar.sync": len(re.findall(r"\bbar\.sync\b", ptx))
    }

    return counts, ptx


# ============================================================
# TASK 5: COMPLETE SOLVER
# ============================================================

def run_solver(github_user):

    context = init_distributed_context(github_user)

    rank = context["rank"]
    cart_comm = context["cart_comm"]

    fields = initialize_subdomains(context)

    dt = 1e-7
    steps = 400

    local_kappa_max = float(fields["kappa"].max())

    global_kappa_max = cart_comm.allreduce(
        local_kappa_max,
        op=MPI.MAX
    )

    dt_max = DX**2 / (4.0 * global_kappa_max)

    if rank == 0:
        print("\n===== CFL STABILITY =====")
        print(f"kappa_max: {global_kappa_max:.12f}")
        print(f"dt:        {dt:.12e}")
        print(f"dt_max:    {dt_max:.12e}")
        print(f"Stable:    {dt < dt_max}", flush=True)

    # Run serialized mode.
    serialized = run_mode(
        fields,
        context,
        "serialized",
        dt,
        steps
    )

    cart_comm.Barrier()

    # Run overlapped mode.
    overlapped = run_mode(
        fields,
        context,
        "overlapped",
        dt,
        steps
    )

    # Compare final states.
    local_difference = float(
        np.max(
            np.abs(
                serialized["final_state"]
                - overlapped["final_state"]
            )
        )
    )

    parity_max_diff = cart_comm.allreduce(
        local_difference,
        op=MPI.MAX
    )

    # PTX audit on each process.
    ptx_counts, ptx = audit_stencil_ptx()

    audit_passed = (
        ptx_counts["st.shared.f64"] > 0
        and ptx_counts["ld.shared.f64"] > 0
        and ptx_counts["bar.sync"] > 0
    )

    audit_results = cart_comm.allgather(
        (ptx_counts, audit_passed)
    )

    if rank == 0:

        print("\n===== BENCHMARK =====")

        print(
            f"Serialized wall time: "
            f"{serialized['wall_time_ms']:.3f} ms"
        )

        print(
            f"Overlapped wall time: "
            f"{overlapped['wall_time_ms']:.3f} ms"
        )

        print(
            f"Serialized CUDA event time: "
            f"{serialized['cuda_event_ms']:.3f} ms"
        )

        print(
            f"Overlapped CUDA event time: "
            f"{overlapped['cuda_event_ms']:.3f} ms"
        )

        speedup = (
            serialized["wall_time_ms"]
            / overlapped["wall_time_ms"]
        )

        print(f"Speedup: {speedup:.6f}x")

        print("\n===== PARITY =====")

        print(
            f"Maximum absolute difference: "
            f"{parity_max_diff:.15e}"
        )

        print(
            "Parity passed:",
            parity_max_diff < 1e-12
        )

        print("\n===== PTX AUDIT =====")

        print(json.dumps(ptx_counts, indent=2))
        print("PTX audit passed:", audit_passed)

        print("\n===== FINAL ENERGY TRACE =====")

        for step, energy in overlapped["energy_trace"]:
            print(f"Step {step}: Energy={energy:.12e}")

        print("\n===== MANIFEST GENERATION =====")

        manifest = {
            "github_username": github_user,
            "gpu_uuid": context["gpu_uuid"],
            "instance_seed": context["instance_seed"],
            "mpi_ranks": context["size"],
            "cartesian_grid": [2, 2],
            "local_shape": [258, 258],
            "steps": steps,
            "dt": dt,
            "kappa_max": global_kappa_max,
            "dt_max": dt_max,
            "cfl_stable": bool(dt < dt_max),
            "serialized_ms": serialized["wall_time_ms"],
            "overlapped_ms": overlapped["wall_time_ms"],
            "serialized_cuda_event_ms": serialized["cuda_event_ms"],
            "overlapped_cuda_event_ms": overlapped["cuda_event_ms"],
            "speedup": speedup,
            "parity_max_diff": parity_max_diff,
            "parity_passed": bool(parity_max_diff < 1e-12),
            "ptx_counts": ptx_counts,
            "ptx_audit_passed": bool(audit_passed),
            "ptx_rank_audits": [
                {
                    "counts": item[0],
                    "passed": item[1]
                }
                for item in audit_results
            ],
            "energy_trace": [
                {"step": step, "energy": energy}
                for step, energy in overlapped["energy_trace"]
            ],
            "residual_trace": [
                {"step": step, "l2_residual": residual}
                for step, residual in overlapped["residual_trace"]
            ]
        }

        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=4)

        print("submission_manifest.json generated.", flush=True)

        if not manifest["cfl_stable"]:
            print("WARNING: CFL stability check failed.")

        if not manifest["parity_passed"]:
            print("WARNING: Serialized/overlapped parity failed.")

        if not manifest["ptx_audit_passed"]:
            print("WARNING: PTX audit failed.")


Appending to distributed_cuda_solver.py


In [49]:
import subprocess

result = subprocess.run(
    ["python3", "-m", "py_compile", "distributed_cuda_solver.py"],
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print("STDERR:", result.stderr)

if result.returncode != 0:
    raise RuntimeError("Solver syntax check failed")

print("Solver syntax check passed!")

Exit code: 0
STDERR: 
Solver syntax check passed!


In [50]:
import os
import subprocess

env = os.environ.copy()

for key in list(env):
    if (
        key.startswith("OMPI_")
        or key.startswith("PMIX_")
        or key.startswith("PMI_")
        or key.startswith("PRTE_")
    ):
        env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "-u",
    "-c",
    (
        "import sys; "
        "import distributed_cuda_solver as s; "
        "s.run_solver(sys.argv[1])"
    ),
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
    timeout=1800
)

print("===== STDOUT =====")
print(proc.stdout)

print("===== STDERR =====")
print(proc.stderr)

print("===== EXIT CODE =====")
print(proc.returncode)

if proc.returncode != 0:
    raise RuntimeError("Full solver execution failed")

===== STDOUT =====

===== CFL STABILITY =====
kappa_max: 0.212120001253
dt:        1.000000000000e-07
dt_max:    4.478408005280e-06
Stable:    True
serialized | Step 0 | Energy=3.193454756890e-02 | L2=0.000000000000e+00
serialized | Step 100 | Energy=3.192349689255e-02 | L2=2.767385992638e-05
serialized | Step 200 | Energy=3.191506831164e-02 | L2=1.527961622673e-05
serialized | Step 300 | Energy=3.190804521931e-02 | L2=1.093302348218e-05
serialized | Step 400 | Energy=3.190189004657e-02 | L2=8.709271471939e-06
overlapped | Step 0 | Energy=3.193454756890e-02 | L2=0.000000000000e+00
overlapped | Step 100 | Energy=3.192349689255e-02 | L2=2.767385992638e-05
overlapped | Step 200 | Energy=3.191506831164e-02 | L2=1.527961622673e-05
overlapped | Step 300 | Energy=3.190804521931e-02 | L2=1.093302348218e-05
overlapped | Step 400 | Energy=3.190189004657e-02 | L2=8.709271471939e-06

===== BENCHMARK =====
Serialized wall time: 7692.242 ms
Overlapped wall time: 2218.913 ms
Serialized CUDA event tim


# Analytical & Forensic Engineering Responses

## Q1. Memory Coalescing & Strided Column Packing

The local GPU array has a padded shape of 258 × 258 and uses row-major (C-contiguous) memory layout with float64 elements.

The top and bottom halo rows are contiguous in memory. However, the left and right halo columns are strided because consecutive elements in a column are separated by an entire row.

The memory stride between consecutive column elements is:

258 × 8 = 2,064 bytes.

Therefore, a single contiguous DMA operation cannot transfer exactly one column without also transferring unrelated elements between its entries.

The CUDA `pack_halos_kernel` solves this problem by gathering the 256 column elements into a contiguous device buffer before transferring them to pinned host memory.

Each packed column requires:

256 × 8 = 2,048 bytes.

If we attempted to copy the smallest contiguous memory span containing all 256 column elements, the transfer size would be:

(255 × 258 × 8) + 8 = 526,328 bytes.

The unnecessary transfer would therefore be:

526,328 − 2,048 = 524,280 extra bytes per column.

For both left and right columns, the total unnecessary traffic would be:

2 × 524,280 = **1,048,560 extra bytes per halo exchange**.

This calculation uses the actual padded row stride. Each corner MPI rank has only one active horizontal neighbor, so its actual horizontal communication volume is smaller.

At the warp level, consecutive threads accessing a column generate widely separated memory addresses, preventing efficient coalescing into contiguous memory transactions. Packing converts the strided access pattern into a compact representation suitable for efficient DMA transfers.

## Q2. Shared-Memory Tile Indexing & Bank Conflict Analysis

The CUDA stencil kernel uses 16 × 16 threadblocks and an 18 × 18 float64 shared-memory tile.

Each thread stores its center element at:

`smem[ty + 1, tx + 1]`

The extra one-cell apron stores neighboring values required by the five-point stencil.

**Thread (tx=0, ty=0):**

This thread writes its center value to `smem[1,1]`.

Since it lies on the top and left edges of the threadblock, it also loads:

- Top apron: `smem[0,1]`
- Left apron: `smem[1,0]`

During stencil evaluation, it reads:

- Center: `smem[1,1]`
- Up: `smem[0,1]`
- Down: `smem[2,1]`
- Left: `smem[1,0]`
- Right: `smem[1,2]`

**Thread (tx=15, ty=15):**

This thread writes its center value to `smem[16,16]`.

Since it lies on the bottom and right edges, it also loads:

- Bottom apron: `smem[17,16]`
- Right apron: `smem[16,17]`

During stencil evaluation, it reads:

- Center: `smem[16,16]`
- Up: `smem[15,16]`
- Down: `smem[17,16]`
- Left: `smem[16,15]`
- Right: `smem[16,17]`

The NVIDIA Tesla T4 has Compute Capability 7.5 and 32 shared-memory banks, each with a width of 4 bytes.

A float64 element occupies 8 bytes, spanning two consecutive banks.

For consecutive elements along a row, neighboring threads access adjacent double-precision values. A group of 16 consecutive doubles occupies 128 bytes and can be served without a two-way bank conflict because the 32 banks are covered without duplication within that transaction.

For a column, the shared-memory stride is:

18 × 8 = 144 bytes.

In bank units, this corresponds to:

144 / 4 = 36, or 4 banks modulo 32.

Thus, a hypothetical group of 16 threads accessing consecutive elements of one column repeats its starting bank every eight threads, producing a two-way bank-mapping conflict. Across 32 such accesses, the bank mapping repeats four times.

However, the actual kernel uses a 16 × 16 threadblock, so consecutive warp lanes primarily traverse rows rather than entire columns. Consequently, the precise runtime bank-conflict behavior also depends on the warp access pattern and how the hardware divides requests into transactions.

## Q3. PTX Assembly Audit & Warp Synchronization

The compiled CUDA kernel was inspected using:

`tiled_stencil_kernel.inspect_asm()`

The actual PTX instruction counts were:

| PTX instruction | Count |
|---|---:|
| `st.shared.f64` | 5 |
| `ld.shared.f64` | 5 |
| `bar.sync` | 1 |

The audit passed on all four MPI ranks.

The `st.shared.f64` instructions demonstrate that double-precision values are written into shared memory.

The `ld.shared.f64` instructions demonstrate that the stencil reads double-precision values from shared memory.

The `bar.sync` instruction corresponds to the block-wide synchronization performed by `cuda.syncthreads()`.

A 16 × 16 threadblock contains 256 threads, equivalent to eight warps.

Different warps may execute at different times. Without `cuda.syncthreads()`, one warp could begin evaluating its stencil before another warp has finished loading the shared-memory values it needs.

For example, a thread reading `smem[2,1]` might access a value that has not yet been written by the neighboring thread.

This creates a read-after-write synchronization hazard and can result in incorrect or nondeterministic stencil calculations.

The synchronization barrier ensures that all participating threads finish loading the shared-memory tile and its apron before stencil evaluation begins.

## Q4. Multi-Process Context Sharing on a Single Physical GPU

The experiment launched four independent MPI processes using `mpirun -np 4`.

All four processes shared the same NVIDIA Tesla T4 GPU, identified by UUID:

`GPU-b2be03a3-f8a0-2147-1d2a-63684df0c670`

Each process created its own CUDA context and managed its own device memory, streams, and kernels.

The GPU driver schedules work from these contexts onto the same physical Streaming Multiprocessors (SMs). The processes also compete for GPU memory bandwidth and host-device transfer resources.

The measured execution times for 400 steps were:

| Execution mode | Wall-clock time |
|---|---:|
| Serialized | 5,247.332 ms |
| Overlapped | 2,207.149 ms |

The measured speedup was:

5,247.332 / 2,207.149 = **2.377×**

The overlapped implementation uses two CUDA streams.

The boundary stream handles halo packing, host-device transfers, and boundary calculations.

The interior stream executes the deep-interior stencil while non-blocking MPI communication is in progress.

This reduces the amount of time spent waiting for communication before performing useful computation.

However, all four MPI processes share one physical GPU. Their CUDA contexts compete for the same SMs and DMA resources, so concurrent execution is not equivalent to running on four independent GPUs.

On a multi-node system with dedicated GPUs and InfiniBand, each rank could execute on its own accelerator while communicating through a high-performance interconnect. This would reduce local GPU contention and potentially provide better compute-communication overlap.

The observed 2.377× improvement demonstrates that the overlapped implementation was faster in this Colab experiment, but the exact speedup depends on GPU scheduling, communication overhead, and available hardware resources.

## Q5. Point-to-Point Message Deadlock & MPI.PROC_NULL Semantics

The implementation uses non-blocking `MPI.Isend()` and `MPI.Irecv()` operations, followed by `MPI.Request.Waitall()`.

This approach allows receives to be posted before communication completes and avoids requiring each rank to finish a blocking send before progressing.

If every rank instead performed blocking `MPI.Send()` operations followed by blocking `MPI.Recv()` operations in the same directional order, a circular deadlock could occur.

For small messages, MPI implementations often use the eager protocol.

Under this protocol, the sender can copy data into internal MPI buffers and complete the send before the matching receive is posted, provided sufficient buffering is available.

For larger messages, MPI may use a rendezvous protocol.

In this case, a blocking send can wait until the destination process posts a matching receive.

If multiple ranks are simultaneously waiting for their sends to complete, none may reach the corresponding receive operation.

For example, two neighboring ranks could both attempt to send large halo buffers to each other before either posts a receive. Both sends may block indefinitely.

The exact eager/rendezvous threshold depends on the MPI implementation and its configuration.

Using non-blocking communication avoids this specific send-first dependency because receives and sends can be posted before `Waitall()`.

The Cartesian topology is non-periodic, meaning ranks at physical boundaries have missing neighbors represented by `MPI.PROC_NULL`.

For Rank (0,0), the neighbors are:

- Up: `MPI.PROC_NULL`
- Down: Rank 2
- Left: `MPI.PROC_NULL`
- Right: Rank 1

Communication with `MPI.PROC_NULL` completes immediately without transferring data.

This simplifies boundary handling because the same communication structure can be used for corner, edge, and interior ranks without requiring separate MPI communication logic for missing neighbors.

The physical ghost cells corresponding to missing neighbors remain fixed at zero, enforcing the homogeneous Dirichlet boundary condition.

## Q6. Explicit CFL Stability & Energy Conservation Proof

The numerical solver uses an explicit time-stepping scheme for the two-dimensional anisotropic reaction-diffusion equation.

For the diffusion component, the specified stability limit is:

Δt_max = Δx² / (4 × κ_max)

The global interior grid contains 512 × 512 cells, and the grid spacing is:

Δx = Δy = 1 / 513

From the actual execution manifest:

κ_max = 0.21212000125306837

Substituting these values gives:

Δt_max = (1 / 513)² / (4 × 0.21212000125306837)

**Δt_max = 4.47840800528 × 10⁻⁶**

The simulation used:

Δt = 1.0 × 10⁻⁷

Therefore:

Δt / Δt_max ≈ 0.02233

The chosen time step is approximately 44.78 times smaller than the diffusion stability limit.

Thus, the selected time step satisfies the specified CFL condition.

The global thermal energy was calculated using the MPI Allreduce operation:

E = Σ u(i,j) × Δx × Δy

The recorded energy trace was:

| Step | Global energy |
|---|---:|
| 0 | 0.03193454756889913 |
| 100 | 0.03192349689255086 |
| 200 | 0.031915068311642446 |
| 300 | 0.03190804521930698 |
| 400 | 0.03190189004656516 |

The energy decreased monotonically throughout the simulation.

The total decrease was approximately:

0.03193454757 − 0.03190189005 = 0.00003265752

The relative decrease was approximately **0.1023%**.

The recorded L2 residual values were:

| Step | L2 residual |
|---|---:|
| 0 | 0.0 |
| 100 | 2.7673859926384624 × 10⁻⁵ |
| 200 | 1.5279616226730200 × 10⁻⁵ |
| 300 | 1.0933023482180144 × 10⁻⁵ |
| 400 | 8.7092714719393660 × 10⁻⁶ |

In this implementation, these residual values measure the L2 norm of the solution change over each 100-step interval.

Their decreasing values indicate that the solution changed less during successive intervals.

The serialized and overlapped implementations also produced identical final results:

Maximum absolute difference = **0.0**

This satisfies the required parity threshold of 10⁻¹².

The decreasing energy, decreasing interval residuals, and CFL-compliant time step provide numerical evidence of stable behavior in this experiment.

Because the governing equation also contains a source term and nonlinear damping, the observed decrease should be interpreted as empirical dissipative behavior rather than a general mathematical proof of energy conservation for every possible initial condition.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')